# 02 - Feature Engineering & Selection

**Deliverables**: `features/{train,val,test}.parquet`, `features/feature_importance.csv`, `reports/feature_analysis.md`

In [7]:
import sys
sys.path.insert(0, '../')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
from data.feature_engineering import build_feature_pipeline, save_features
clean_df = pd.read_parquet('../data/clean_network_logs.parquet')
print(f'Clean data: {clean_df.shape}')

Clean data: (9437, 15)


## 1. Build Feature Pipeline

In [9]:
feature_data = build_feature_pipeline(clean_df)
X_train = feature_data['X_train'].fillna(0)
y_train = feature_data['y_train']
X_val = feature_data['X_val'].fillna(0)
y_val = feature_data['y_val']
X_test = feature_data['X_test'].fillna(0)
y_test = feature_data['y_test']
print(f'Features: {len(feature_data["feature_cols"])}')
print(f'Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}')
print(f'Malicious rate — train={y_train.mean():.3f}, val={y_val.mean():.3f}, test={y_test.mean():.3f}')

Features: 36
Train=(6275, 36), Val=(1614, 36), Test=(1548, 36)
Malicious rate — train=0.049, val=0.048, test=0.050


## 2. Statistical Feature Selection

In [10]:
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({'feature': X_train.columns, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)

f_scores, p_values = f_classif(X_train, y_train)
f_df = pd.DataFrame({'feature': X_train.columns, 'f_score': f_scores, 'p_value': p_values}).sort_values('f_score', ascending=False)

corrs = []
for col in X_train.columns:
    rho, _ = spearmanr(X_train[col], y_train)
    corrs.append({'feature': col, 'spearman_rho': rho})
corr_df = pd.DataFrame(corrs).sort_values('spearman_rho', key=abs, ascending=False)

print('Top 15 MI:'); print(mi_df.head(15).to_string(index=False))
print('\nTop 15 F:'); print(f_df.head(15).to_string(index=False))

Top 15 MI:
               feature  mi_score
           day_of_week  0.005763
      unique_ports_cum  0.005365
           src_ip_freq  0.004802
    bytes_clean_max_1H  0.003521
        conn_count_cum  0.003056
                 month  0.002922
                 bytes  0.002489
           bytes_clean  0.002438
  bytes_clean_max_5min  0.002026
 bytes_clean_count_24H  0.001556
  bytes_clean_count_1H  0.001532
bytes_clean_median_24H  0.001472
   bytes_clean_min_24H  0.000799
   bytes_clean_mean_1H  0.000616
  bytes_clean_std_5min  0.000380

Top 15 F:
             feature  f_score  p_value
               month 6.296853 0.012120
               bytes 4.783378 0.028773
         bytes_clean 4.299808 0.038157
                hour 1.570021 0.210251
         src_ip_freq 1.566614 0.210746
       protocol_freq 1.555576 0.212361
          is_weekend 1.264816 0.260784
         day_of_week 1.255534 0.262541
    protocol_encoded 1.226936 0.268046
status_clean_encoded 1.184281 0.276528
      conn_count_cum 

/home/user/.local/lib/python3.10/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 33] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/user/.local/lib/python3.10/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/tmp/ipykernel_2397834/3734147073.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(X_train[col], y_train)


## 3. Model-Based Importance

In [11]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_imp = pd.DataFrame({'feature': X_train.columns, 'rf_importance': rf.feature_importances_}).sort_values('rf_importance', ascending=False)

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0)
xgb.fit(X_train, y_train)
xgb_imp = pd.DataFrame({'feature': X_train.columns, 'xgb_importance': xgb.feature_importances_}).sort_values('xgb_importance', ascending=False)

print('Top 15 RF:'); print(rf_imp.head(15).to_string(index=False))
print('\nTop 15 XGB:'); print(xgb_imp.head(15).to_string(index=False))

Top 15 RF:
             feature  rf_importance
         bytes_clean       0.207059
               bytes       0.195580
        day_of_month       0.108388
                hour       0.107210
           port_freq       0.075652
               month       0.068474
status_clean_encoded       0.064946
       protocol_freq       0.054182
         day_of_week       0.053952
    protocol_encoded       0.046922
          is_weekend       0.014494
         src_ip_freq       0.002391
    unique_ports_cum       0.000276
time_since_last_conn       0.000182
         dst_ip_freq       0.000102

Top 15 XGB:
             feature  xgb_importance
           port_freq        0.102712
               month        0.100846
               bytes        0.100125
         bytes_clean        0.098202
        day_of_month        0.097270
    protocol_encoded        0.097092
       protocol_freq        0.097041
                hour        0.095764
         day_of_week        0.093217
status_clean_encoded        0.

## 4. Multicollinearity & VIF

In [12]:
pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [13]:
corr_matrix = X_train.corr()
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        c = corr_matrix.iloc[i, j]
        if abs(c) > 0.9:
            high_corr.append({'f1': corr_matrix.columns[i], 'f2': corr_matrix.columns[j], 'corr': c})
high_corr_df = pd.DataFrame(high_corr).sort_values('corr', key=abs, ascending=False) if high_corr else pd.DataFrame()
print(f'High-correlation pairs (|r| > 0.9): {len(high_corr_df)}')

from statsmodels.stats.outliers_influence import variance_inflation_factor
sample = X_train.sample(n=min(1000, len(X_train)), random_state=42)
vif_data = pd.DataFrame({'feature': sample.columns,
                         'VIF': [variance_inflation_factor(sample.values, i) for i in range(sample.shape[1])]})
vif_data = vif_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['VIF']).sort_values('VIF', ascending=False)
print(f'Max VIF: {vif_data["VIF"].max():.1f}, features with VIF > 10: {(vif_data["VIF"] > 10).sum()}')
print(vif_data.head(10).to_string(index=False))

High-correlation pairs (|r| > 0.9): 4


/tmp/ipykernel_2397834/3352927324.py:14: UserWarning: The design matrix is poorly conditioned (condition number=7.37e+82). VIF calculations may be numerically unstable.
  'VIF': [variance_inflation_factor(sample.values, i) for i in range(sample.shape[1])]})
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.l

Max VIF: 1000799917193443.5, features with VIF > 10: 5
             feature          VIF
    unique_ports_cum 1.000800e+15
      conn_count_cum 1.000800e+15
      unique_dst_cum 1.000800e+15
               bytes 8.880324e+01
         bytes_clean 8.871518e+01
time_since_last_conn 3.000535e+00
         day_of_week 2.588303e+00
          is_weekend 2.582101e+00
         src_ip_freq 1.419044e+00
         dst_ip_freq 1.068919e+00


/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python3.10/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/home/user/.local/lib/python

## 5. Save Features + Report

In [14]:
importance_df = (mi_df.merge(f_df[['feature', 'f_score', 'p_value']], on='feature')
                        .merge(corr_df, on='feature')
                        .merge(rf_imp, on='feature')
                        .merge(xgb_imp, on='feature'))
rank_cols = ['mi_score', 'f_score', 'spearman_rho', 'rf_importance', 'xgb_importance']
for c in rank_cols:
    importance_df[f'{c}_rank'] = importance_df[c].rank(ascending=False)
importance_df['mean_rank'] = importance_df[[f'{c}_rank' for c in rank_cols]].mean(axis=1)
importance_df = importance_df.sort_values('mean_rank')
importance_df.to_csv('../features/feature_importance.csv', index=False)

save_features(Path('../features'), feature_data)

lines = ['# Feature Analysis Report', '',
         '## Summary',
         f'- Total features created: {len(feature_data["feature_cols"])}',
         f'- Feature version: {feature_data["feature_version"]}',
         f'- Train / Val / Test: {feature_data["X_train"].shape[0]:,} / {feature_data["X_val"].shape[0]:,} / {feature_data["X_test"].shape[0]:,}',
         '', '## Top 20 Features by Composite Rank']
for rank, (_, row) in enumerate(importance_df.head(20).iterrows(), start=1):
    lines.append(f'{rank}. **{row["feature"]}**: MI={row["mi_score"]:.4f}, F={row["f_score"]:.1f}, rho={row["spearman_rho"]:+.3f}, mean_rank={row["mean_rank"]:.1f}')
lines += [
    '', '## Multicollinearity',
    f'- High-correlation pairs (|r| > 0.9): {len(high_corr_df)}',
    f'- Max VIF: {vif_data["VIF"].max():.1f}',
    f'- Features with VIF > 10: {(vif_data["VIF"] > 10).sum()}',
    '', '## Recommendations',
    '- Keep top-50 by composite rank; drop high-VIF pairs.',
    '- Rolling-window features capture per-source behavioral patterns.',
    '- Encoded categoricals (protocol, status) add signal.',
]
Path('../reports').mkdir(parents=True, exist_ok=True)
Path('../reports/feature_analysis.md').write_text('\n'.join(lines), encoding='utf-8')
print('feature_analysis.md saved')

feature_analysis.md saved
